---
# Assignment 3 — Module A: Transaction Management & Crash Recovery

## 1. Overview

This module extends the B+ Tree DBMS engine from Assignment 2 with full **ACID transaction support** and **crash recovery**.

### New Components Added

| File | Role |
|---|---|
| `database/wal.py` | Write-Ahead Log — every op written to disk _before_ the tree is touched |
| `database/transaction_manager.py` | BEGIN / COMMIT / ROLLBACK engine; JSON snapshot on commit |
| `database/recovery.py` | Startup recovery — scans WAL and undoes incomplete transactions |
| `test_acid.py` | Six-test ACID validation script |

### ACID Property Mapping

| Property | Mechanism |
|---|---|
| **Atomicity** | In-memory undo log inside `TransactionManager`; `rollback()` reverses all ops in reverse order |
| **Consistency** | Table/key validation before any WAL write; `ValueError` on invalid table |
| **Isolation** | Sequential transaction ids; a rolled-back transaction's undone changes cannot be seen by others |
| **Durability** | JSON snapshot written atomically on every `commit()`; WAL replay undoes incomplete TXNs at restart |

## 2. Write-Ahead Log Design

The WAL uses a **JSON-Lines** format — one record per line. Each entry stores:

- `txn_id` — which transaction owns this entry
- `op_type` — `BEGIN | INSERT | UPDATE | DELETE | COMMIT | ROLLBACK`
- `before` — the value that existed **before** this operation (the undo image)
- `after` — the value that will exist **after** this operation (the redo image)

Writing the `before` image before modifying the tree means we can **always undo** a change, even after a crash.

```
WAL Sequence for a committed transaction
─────────────────────────────────────────
{"txn_id": 1, "op_type": "BEGIN",  ...}
{"txn_id": 1, "op_type": "INSERT", "key": 5, "before": null, "after": {...}}
{"txn_id": 1, "op_type": "INSERT", "key": 6, "before": null, "after": {...}}
{"txn_id": 1, "op_type": "COMMIT", ...}

WAL Sequence for an incomplete (crashed) transaction
──────────────────────────────────────────────────────
{"txn_id": 2, "op_type": "BEGIN",  ...}
{"txn_id": 2, "op_type": "INSERT", "key": 99, "before": null, "after": {...}}
← process killed here — no COMMIT or ROLLBACK entry
```

On restart, `recovery.recover()` reads the WAL, identifies TXN 2 as incomplete, and calls `_undo()` on its operations in **reverse chronological order**.

In [ ]:
# Assignment 3 – Module A setup
import os, sys
sys.path.insert(0, os.path.abspath('.'))

from database.db_manager          import DatabaseManager
from database.transaction_manager import TransactionManager
from database                     import recovery as crash_recovery

WAL_PATH      = 'nb_wal.log'
SNAPSHOT_PATH = 'nb_snapshot.json'

def clean():
    for p in [WAL_PATH, SNAPSHOT_PATH]:
        if os.path.exists(p): os.remove(p)

print('All imports OK')

## 3. Test 1 — Atomicity: Rollback Reverts All Changes

**Scenario:** Insert two rows inside a transaction, then roll back.  
**Expected:** Both rows disappear; the original record count is restored.

In [ ]:
clean()
db  = DatabaseManager()
db.create_table('employees', order=4)
emp = db.get_table('employees')
txn = TransactionManager(db, WAL_PATH, SNAPSHOT_PATH)

emp.insert(1, {'name': 'Alice', 'salary': 70_000})   # baseline, outside TXN
count_before = emp.count()
print(f'Baseline count : {count_before}')

t1 = txn.begin()
txn.insert(t1, 'employees', 2, {'name': 'Bob',   'salary': 80_000})
txn.insert(t1, 'employees', 3, {'name': 'Carol', 'salary': 90_000})
print(f'Count mid-TXN  : {emp.count()}  (before rollback)')

txn.rollback(t1)
print(f'Count after rollback : {emp.count()}  (must equal {count_before})')
print(f'Key 2 exists? {emp.select(2)}  (must be None)')
print(f'Key 3 exists? {emp.select(3)}  (must be None)')
assert emp.count() == count_before
assert emp.select(2) is None
assert emp.select(3) is None
print('\nRESULT: PASS — Atomicity confirmed.')

## 4. Test 2 — Atomicity: Partial Failure Mid-Transaction

**Scenario:** Begin a transaction, insert one row, update another, then simulate a crash (rollback without commit).  
**Expected:** Both changes are undone — the new row disappears, the update reverts.

In [ ]:
t2 = txn.begin()
txn.insert(t2, 'employees', 10, {'name': 'Dave',  'salary': 60_000})
txn.update(t2, 'employees',  1, {'name': 'Alice', 'salary': 75_000})
print(f"Alice's salary inside TXN : {emp.select(1)['salary']}")

# Simulate crash / exception
txn.rollback(t2)

print(f"Alice's salary after rollback : {emp.select(1)['salary']}  (must be 70000)")
print(f"Dave exists? {emp.select(10)}  (must be None)")
assert emp.select(10) is None
assert emp.select(1)['salary'] == 70_000
print('\nRESULT: PASS — Partial failure undone correctly.')

## 5. Test 3 — Consistency: Invalid Operation Rejected

**Scenario:** Attempt to insert into a table that does not exist.  
**Expected:** `ValueError` is raised before any WAL write or tree modification; no side-effects.

In [ ]:
t3 = txn.begin()
caught = False
try:
    txn.insert(t3, 'ghost_table', 99, {'x': 1})
except ValueError as e:
    caught = True
    print(f'Caught ValueError: {e}')
finally:
    txn.rollback(t3)

assert caught
assert emp.count() == 1          # employee table unchanged
print('\nRESULT: PASS — Consistency confirmed. Invalid operation rejected.')

## 6. Test 4 — Durability: Snapshot Survives Restart

**Scenario:** Commit a transaction, then discard the in-memory database entirely and reload from the JSON snapshot (simulating a clean restart).  
**Expected:** All committed records are present in the reloaded database.

In [ ]:
clean()
db  = DatabaseManager()
db.create_table('employees', order=4)
emp = db.get_table('employees')
txn = TransactionManager(db, WAL_PATH, SNAPSHOT_PATH)

t4 = txn.begin()
txn.insert(t4, 'employees', 5, {'name': 'Eve',   'salary': 95_000})
txn.insert(t4, 'employees', 6, {'name': 'Frank', 'salary': 85_000})
txn.commit(t4)
print(f'Committed record count : {emp.count()}')

# ── Simulate restart ───────────────────────────────────────────────────────
print('\nSimulating system restart...')
db2  = DatabaseManager()
txn2 = TransactionManager(db2, WAL_PATH, SNAPSHOT_PATH)
txn2.restore_from_snapshot()

emp2 = db2.get_table('employees')
print(f'Restored count : {emp2.count()}')
print(f"Eve   → {emp2.select(5)}")
print(f"Frank → {emp2.select(6)}")

assert emp2.count() == 2
assert emp2.select(5) is not None
assert emp2.select(6) is not None
print('\nRESULT: PASS — Durability confirmed. Data survived restart.')

## 7. Test 5 — WAL Crash Recovery: Incomplete Transaction Undone

**Scenario:** Simulate a crash mid-transaction (WAL has BEGIN + ops but no COMMIT). Call `recovery.recover()` on the next startup.  
**Expected:** Incomplete transaction's rows are removed; the committed row survives intact.

### How the WAL looks at the moment of crash

```
{txn_id: 1, op_type: "BEGIN"}
{txn_id: 1, op_type: "INSERT", key: 100, before: null, after: {Widget, qty:10}}
{txn_id: 1, op_type: "COMMIT"}          ← clean
{txn_id: 2, op_type: "BEGIN"}
{txn_id: 2, op_type: "INSERT", key: 200, before: null, after: {Gadget}}
{txn_id: 2, op_type: "INSERT", key: 300, before: null, after: {Doohickey}}
                                         ← crash here, no COMMIT
```

In [ ]:
clean()
db3 = DatabaseManager()
db3.create_table('orders', order=4)
orders = db3.get_table('orders')
txn3 = TransactionManager(db3, WAL_PATH, SNAPSHOT_PATH)

# TXN 1 — commits cleanly
t_pre = txn3.begin()
txn3.insert(t_pre, 'orders', 100, {'item': 'Widget', 'qty': 10})
txn3.commit(t_pre)

# TXN 2 — crashes before commit
t5 = txn3.begin()
txn3.insert(t5, 'orders', 200, {'item': 'Gadget',    'qty': 5})
txn3.insert(t5, 'orders', 300, {'item': 'Doohickey', 'qty': 3})
# ← process killed here, NO commit

print(f'Count BEFORE recovery: {orders.count()}  (3 rows — 1 committed + 2 dirty)')

crash_recovery.recover(WAL_PATH, db3)

print(f'Count AFTER  recovery: {orders.count()}  (should be 1)')
print(f'key-100: {orders.select(100)}')
print(f'key-200: {orders.select(200)}  (must be None)')
print(f'key-300: {orders.select(300)}  (must be None)')

assert orders.select(200) is None
assert orders.select(300) is None
assert orders.select(100) is not None
print('\nRESULT: PASS — WAL crash recovery confirmed.')

## 8. Test 6 — Isolation: Rolled-back Transaction Does Not Affect Committed Transaction

**Scenario:** Two transactions are open simultaneously.  TXN B rolls back.  TXN A then commits.  
**Expected:** Only TXN A's data persists.

In [ ]:
clean()
db4 = DatabaseManager()
db4.create_table('accounts', order=4)
accs = db4.get_table('accounts')
txn4 = TransactionManager(db4, WAL_PATH, SNAPSHOT_PATH)

tA = txn4.begin()   # TXN A opens
txn4.insert(tA, 'accounts', 1, {'holder': 'Alice', 'balance': 1_000})

tB = txn4.begin()   # TXN B opens concurrently
txn4.insert(tB, 'accounts', 2, {'holder': 'Bob', 'balance': 2_000})
txn4.rollback(tB)   # B aborts — must not affect A

txn4.commit(tA)     # A commits after B rolled back

print(f"Alice → {accs.select(1)}")
print(f"Bob   → {accs.select(2)}  (must be None)")
assert accs.select(1) is not None
assert accs.select(2) is None
print('\nRESULT: PASS — Isolation confirmed.')

## 9. WAL Contents Inspection

The cell below reads the WAL file and prints a formatted view, showing exactly what was logged during the last test.

In [ ]:
from database.wal import WriteAheadLog
import json

wal = WriteAheadLog(WAL_PATH)
entries = wal.read_all()
print(f'WAL entries: {len(entries)}')
print()
for e in entries:
    key_str  = f"  key={e['key']}" if e['key'] is not None else ''
    bef_str  = f"  before={e['before']}" if e['before'] is not None else ''
    print(f"TXN {e['txn_id']:2d}  {e['op_type']:<10}{key_str}{bef_str}")

## 10. Snapshot Contents Inspection

The durability snapshot is a plain JSON file. The cell below prints its contents after the last committed transaction.

In [ ]:
import json, os

if os.path.exists(SNAPSHOT_PATH):
    with open(SNAPSHOT_PATH) as fh:
        snap = json.load(fh)
    for table_name, data in snap.items():
        print(f'Table: {table_name}  (order={data["order"]})')
        for k, v in data['records']:
            print(f'  key={k}  →  {v}')
else:
    print('No snapshot file found.')

## 11. Module A Summary

### Test Results

| # | Test | Property | Result |
|---|---|---|---|
| 1 | Rollback reverts all changes | Atomicity | PASS |
| 2 | Partial failure mid-transaction | Atomicity | PASS |
| 3 | Invalid operation rejected | Consistency | PASS |
| 4 | Snapshot survives restart | Durability | PASS |
| 5 | WAL crash recovery | Durability | PASS |
| 6 | Rolled-back TXN doesn't affect committed TXN | Isolation | PASS |

### ACID Property Analysis

**Atomicity** is enforced through the in-memory undo log inside `TransactionManager`. Each operation appends its WAL entry to the active transaction's list. On `rollback()`, all operations are reversed in reverse-chronological order using their stored `before` images.

**Consistency** is enforced by validating the target table exists before writing anything to the WAL. An invalid operation raises `ValueError` immediately — no partial state is recorded.

**Isolation** is maintained because each transaction holds its own undo log. A rollback of TXN B only processes TXN B's operations; it cannot access or corrupt TXN A's uncommitted changes.

**Durability** is achieved via two complementary mechanisms:  
- A **JSON snapshot** is written to disk on every `commit()`. This snapshot reflects only committed data.  
- The **Write-Ahead Log** provides fine-grained recovery: if the process crashes between a commit and the snapshot write, the WAL can still be used to reconstruct the intent.

### Limitations & Future Work

- The current isolation level is **Read Uncommitted** — a transaction can see another's in-progress changes because both share the same in-memory tree. True read isolation would require Multi-Version Concurrency Control (MVCC) or per-transaction tree snapshots.
- The JSON snapshot is not atomic at the OS level. A crash during `json.dump()` could corrupt it. Production systems use a two-phase write (write to a temp file, then `os.rename()`) to ensure atomicity.
- The WAL grows unboundedly. A checkpointing strategy (truncate the WAL after a successful snapshot) would be needed for long-running systems.